In [1]:
import os, sys
from pathlib import Path
import pandas as pd
import subprocess

In [2]:
sys.path.append("../../../training_data")

In [3]:
from utils.utils import Cif

In [4]:
with open("../../../training_data/8.Apos/Extra_set/features.pkl", "rb") as f:
    extras_featuresd = pd.read_pickle(f)

len(extras_featuresd), extras_featuresd

(9,
 {'8sgj':     Residues                                                          \
           pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
  0       8sgj               1             A           52            A   
  1       8sgj               1             A           53            A   
  2       8sgj               1             A           54            A   
  3       8sgj               1             A           55            A   
  4       8sgj               1             A           56            A   
  ..       ...             ...           ...          ...          ...   
  746     8sgj               1             A          941            A   
  747     8sgj               1             A          942            A   
  748     8sgj               1             A          943            A   
  749     8sgj               1             A          944            A   
  750     8sgj               1             A          945            A   
  
                       

# Make predictions

Edits throughout to fix:
- Hardcoded paths
- Pass locations of ProtT5 and the nr database

In [ ]:
for pdb, feats in extras_featuresd.items():
        
    path = Path("../../other_tools/AlloFusion/AlloFusion/Case Study").resolve()
    path.mkdir(exist_ok = True)
    try:
        outdir = Path(pdb)
        if not outdir.exists():
            print(pdb)
            
            chain = feats[('Residues', 'auth_asym_id')].unique().item()
            
            origpdbf = Path(f"../structures/{pdb}.pdb").resolve()
    
            seq = (
                pd.DataFrame(
                    Cif(pdb, origpdbf.with_suffix(".cif")).cif.data["_entity_poly"], dtype=str
                )
                .query(f"entity_id == '{feats[('Residues', 'label_entity_id')].unique().item()}'")
                ["pdbx_seq_one_letter_code_can"].item()
                .replace("\n", "")
            )
            
            pdbf = path / f"{pdb}.pdb"
            if not pdbf.exists():
                pdbf.symlink_to(origpdbf)
        
            subprocess.run(f"python AlloFusionMain.py --PDBID {pdb} --CHAIN {chain} --SEQ {seq} --huggingface_dir /data/fnerin/huggingface --nr_database /data/fnerin/nr_database/nr", cwd=path.parent, shell=True, check=True)

            path.rename(outdir)
            
    except Exception as e:
        print(f"ERROR: ", pdb)
        print(e)
        path.rename(f"{pdb}_error")
        continue

AF-A0A1D8PQM9-F1


2025-10-10 19:43:59.681826: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-10 19:43:59.698192: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and

# Process

In [5]:
results = {}

for pdb, feats in extras_featuresd.items():
    chain = feats[('Residues', 'auth_asym_id')].unique().item()
    resf = f"{pdb}/{pdb}_allosteric_residues.txt"
    if os.path.isfile(resf):
        with open(resf) as f:
            txt = f.read()
        chain = txt.split("Chain", 1)[1].strip().split()[0]
        resids = [x for x in txt.split("resid", 1)[1].replace("(", "").replace(")", "").replace(",", " ").split() if x.isdigit()]   

        results[pdb.lower()] = {"pocket": {"residues": (
            pd.DataFrame({"auth_asym_id": [chain]*len(resids), "auth_seq_id": resids}, dtype=str)
            .merge(Cif(pdb, f"../structures/{pdb}.cif").residues)
            [["auth_asym_id", "auth_seq_id"]]
        )}}

len(results), results

(9,
 {'8sgj': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         451
    1            A         810}},
  'af-a0a1d8pqm9-f1': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         484}},
  '7l6r': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A        6878
    1            A        6880
    2            A        6968
    3            A        7000}},
  '8vw5': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         125
    1            A         132
    2            A         143
    3            A         151
    4            A         171
    5            A         289
    6            A         297
    7            A         308}},
  '6yhr': {'pocket': {'residues':    auth_asym_id auth_seq_id
    0             A         577
    1             A         581
    2             A         704
    3             A         713
    4             A         720
    5             A         721
    6    

In [6]:
pd.to_pickle(results, "allofusion_results.pkl")